# Zgłaszanie błędów

W Pythonie możemy zgłosić błąd składnią `raise RodzajBłędu(komunikat_o_błędzie_dla_użytkownika)`. Komunikat dla użytkownika jest opcjonalny. Wszystkie wbudowane rodzaje błędów w Pythonie znajdziesz w dokumentacji https://docs.python.org/3/library/exceptions.html#exception-hierarchy, wiele edytorów kodu podpowie też ich nazwy.

In [1]:
raise ValueError('Zgłaszam błąd.')

ValueError: Zgłaszam błąd.

Zgłoszenie błedu przerywa program.

In [2]:
raise RuntimeError()
print('Ten kod się nie wykona')

RuntimeError: 

Można też tworzyć własne typy błędów, ale wymaga to dziedziczenia klas, które będzie omówione na przyszlych zajęciach.

# Metody specjalne/magincze

Python pozwala na implementowanie w klasach operacji na obiektach, takich jak dodawanie obiektów znakiem `+`, konwersja typów, wypisanie jako napis, branie elementu `obiekt[klucz]` itp. Robi się to, definiując w klasie specjalnie nazwane metody, obsługujące te czynności.

Na wcześniejszych zajęciach omówiliśmy już metodę specjalną `__init__(self, arguemnty)`, która definiuje konstruktor klasy. Oto kilka innych, przydatnych metod magicznych:

`__str__(self)` - opisuje konwersję obiektu do napisu. Wywoływana na prykład, gdy piszemy `print(obiekt)`. Powinna zwrócić stringa.

In [3]:
class A():

    def __str__(self):
        return f'To się wyprintuje, gdy printujemy obiekt klasy {A}'

a = A()
print(a)

To się wyprintuje, gdy printujemy obiekt klasy <class '__main__.A'>


`__float__(self)` - konwersja do liczby zmiennoprzecinkowej. Wywoływana, gdy piszemy `float(obiekt)`. Powinna zwrócić floata.

In [4]:
class B():

    def __float__(self):
        return 1.0
    
a = B()

print(float(a))

1.0


Podobnie działają `__int__`, `__bool__` (wartość logiczna) i `__complex__`.

### Operacje arytmetyczne

`__add__(self, other)` - opisuje dodawanie naszego obiektu (tak jak my rozumiemy, co to znaczy coś do niego dodać). Wywoływana przez kod `obiekt+other`. Powinna zwrócić wynik dodawania obiektów.

In [5]:
class C():

    def __init__(self, napis):
        self.napis = napis

    def __add__(self, other):
        return C(self.napis + other.napis)
    
    def __str__(self):
        return self.napis
    
a = C('Ala')
b = C(' ma kota')

print(a + b)

Ala ma kota


Przy metodach takich jak `__add__(self, other)` często przydaje się sprawdzenie, jakiego rodzaju jest other. Można to zrobić na przykład funkcją `isinstance(obiekt, klasa)`. Funkcja ta zwraca `True`, jeśli `obiekt` należy do klasy `klasa`, a `False` w przeciwnym przypadku.

Można też napisać `isinstance(obiekt, (klasa1, klasa2, klasa3))` i wtedy zwróci `True`, jeśli obiekt należy do **dowolnej** z tych klas.

In [6]:
class D():

    def __init__(self, napis):
        self.napis = napis

    def __add__(self, other):
        if isinstance(other, D):
            return D(self.napis + other.napis)
        elif isinstance(other, (int, float)):
            return D(self.napis + str(other))
        else:
            raise TypeError(f'Nie umiem dodać {other}')

    def __str__(self):
        return self.napis
    
a = D('Ala')
b = D(' ma kota')

print(a + b)
print(a + 3.14)
print(a + [1, 2, 3])

Ala ma kota
Ala3.14


TypeError: Nie umiem dodać [1, 2, 3]

Oprócz `isinstace` Python posiada jeszcze funkcję `type`, która zwraca typ(klasę) obiektu, na którym jest wywoływana.

In [7]:
print(type(1))
print(type([1, 2, 3]))
print(type(a))

<class 'int'>
<class 'list'>
<class '__main__.D'>


Pdobnie do `__add__`, `__mul__` definiuje mnożenie `*`, `__sub__` odejmowanie `-`, a `__truediv__` dzielenie `/`.

Powyższa klasa definiuje, co się dzieje, gdy dodajemy coś **do** jej obiektu, ale nie, gdy jej biekt jest dodawany **do czegoś**.

In [8]:
print(a + 1)
print(1 + a)

Ala1


TypeError: unsupported operand type(s) for +: 'int' and 'D'

Do tego celu służy `__radd__(self, other)`, która obsługuje sytuację `other+self`. `__radd__` jest wywoływana tylko, jeśli `__add__` klasy obiektu `other` nie umie obsłużyć dodawania `other+self`.

In [9]:
class E():

    def __init__(self, napis):
        self.napis = napis

    def __add__(self, other):
        if isinstance(other, E):
            return E(self.napis + other.napis)
        elif isinstance(other, (int, float)):
            return E(self.napis + str(other))
        else:
            raise TypeError(f'Nie umiem dodać {other}')
        
    def __radd__(self, other):
        if isinstance(other, (int, float)): # nie musimy obsługiwać sytuacji, gdy other jest kalsy E, bo metoda __add__ już to umie
            return E(str(other) + self.napis)
        else:
            raise TypeError(f'Nie umiem dodać {other}')

    def __str__(self):
        return self.napis
    
a = E('Ala')

print(a + 123)
print(0.12 + a)

Ala123
0.12Ala


Podobnie działają `__rsub__`, `__rmul` i `__rtrediv__`.

Python posiada też oparacje zwiększenia o, `+=`. Obsługuje ją metoda `__iadd__(self, other)`, która powinna zwrócić wielkość, jaka ma zastąpić `self` na skutek zwiększenia.

In [10]:
class F():

    def __init__(self, napis):
        self.napis = napis

    def __add__(self, other):
        if isinstance(other, F):
            return F(self.napis + other.napis)
        elif isinstance(other, (int, float)):
            return F(self.napis + str(other))
        else:
            raise TypeError(f'Nie umiem dodać {other}')
        
    def __iadd__(self, other):
        return self + other # ponieważ __add__ jest zdefiniowana, możemy użyć `+`, zamiast ją przepisywać

    def __str__(self):
        return self.napis
    
a = F('Ala')
b = F(' ma kota')

a += b
print(a)

Ala ma kota


Analogicznie działają `__isub__`, `__imul__` i `__itruediv__`.

`__neg__(self)` - operacja wzięcia ujemnej wartości, odpowiada kodowi `-self`. Powinna zwrócić tą wartość.

In [11]:
class G():

    def __init__(self, napis):
        self.napis = napis

    def __neg__(self):
        return self.napis[::-1]

    def __str__(self):
        return self.napis
    
a = G('Ala ma kota')
print(-a)

atok am alA


### Operacje porównania

`__lt__(self, other)` - operacja porównania znakiem "mniejsze", wywoływana przez `self<other`. Powinna zwrócić wartość logiczną.

In [12]:
class H():

    def __init__(self, liczba):
        self.liczba = liczba

    def __lt__(self, other):
        if self.liczba < other.liczba:
            return True
        else:
            return False
        
a = H(0)
b = H(212)
print(a<b)
print(b<a)

True
False


Podobne metody zdefiniowane są dla innych operacji porównywania:
| metoda | operacja |
|:------:|:--------:|
| `__lt__` | `<` |
| `__le__` | `<=` |
| `__gt__` | `>` |
| `__ge__` | `>=` |
| `__eq__` | `==` |
| `__ne__` | `!=` |

### Operacje na elementach

`__getitem__(self, key)` - zwróć element dany indeksem/kluczem `key`, wywoływane przez `self[key]`.

`__setitem__(self, key, value)` - przypisz wartość pod kluczem `key`, wywoływane przez `self[key] = value`.

`__delitem__(self, key)` - usuń po kluczu `key`, wywoływane przez `del self[key]`.

`__contains__(self, item)` - czy zawiera `item`, wywoływane przez `key in self`.

`__len__(self)` - zwraca długość `self`, wywołane przez `len(self)`. Powinno zwrócić liczbę całkowitą.

In [13]:
class I():

    def __init__(self, lista):
        self.lista = lista

    def __str__(self):
        return str(self.lista)

    def __getitem__(self, key):
        return self.lista[key]
    
    def __setitem__(self, key, value):
        self.lista[key] = value

    def __delitem__(self, key):
        del self.lista[key]

    def __contains__(self, item):
        return item in self.lista
    
    def __len__(self):
        return len(self.lista)
    
a = I([1, 2, 3])
print(a[0])
a[1] = 'abc'
print(a)
print(3 in a)
del a[-1]
print(a)
print(3 in a)
print(len(a))

1
[1, 'abc', 3]
True
[1, 'abc']
False
2


### Inne

`__call__(self, argumenty)` - wywołanie obiektu jak funkcji, wywołane kodem `self(argumenty)`.

In [14]:
class J():

    def __init__(self):
        self.lista = []
    
    def __call__(self, x):
        self.lista.append(x*2)
        return x*2
    
podwajacz = J()

print(podwajacz(1))
print(podwajacz(3.14))
print(podwajacz('Ala ma kota.'))
print(podwajacz.lista)

2
6.28
Ala ma kota.Ala ma kota.
[2, 6.28, 'Ala ma kota.Ala ma kota.']


Python posiada jeszcze wiele innych metod specjalnych, z których korzysta się rzadziej, nie warto ich wszystkich zapamiętywać.